# 📊 Notebook 1 — Data Pipeline, Preprocessing & EDA

**Dataset**: APTOS 2019 Blindness Detection  
**Goal**: Explore the dataset, demonstrate Ben Graham's preprocessing pipeline, and analyze class distribution, image statistics, and pixel intensities.

---

## 1. Setup & Imports

In [ ]:
import sys
import os
from pathlib import Path

# Add src/ to path so we can import our modules
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from config import (
    APTOS_TRAIN_CSV, APTOS_TRAIN_IMAGES,
    IMAGE_SIZE, DR_GRADES, DR_COLORS, RANDOM_SEED
)
from preprocessing import (
    ben_graham_preprocess, circular_crop, local_color_normalization
)

# Plot style
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print(f'Project root: {PROJECT_ROOT}')
print(f'APTOS images: {APTOS_TRAIN_IMAGES}')

## 2. Load Dataset Metadata

In [ ]:
df = pd.read_csv(APTOS_TRAIN_CSV)
print(f'Total images: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head(10)

## 3. Class Distribution Analysis

The APTOS 2019 dataset is **severely imbalanced** — grade 0 (No DR) dominates, while grade 3 (Severe) and grade 4 (Proliferative DR) are rare. This is the primary motivation for **Focal Loss**.

In [ ]:
class_counts = df['diagnosis'].value_counts().sort_index()
class_pct = (class_counts / len(df) * 100).round(1)

print('Class Distribution:')
print('=' * 45)
for grade, count in class_counts.items():
    name = DR_GRADES[grade]
    pct = class_pct[grade]
    print(f'  Grade {grade} ({name:20s}): {count:5d}  ({pct:5.1f}%)')
print(f'  {"Total":26s}: {len(df):5d}')

# Imbalance ratio
max_count = class_counts.max()
min_count = class_counts.min()
print(f'\nImbalance ratio (max/min): {max_count/min_count:.1f}x')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = [DR_COLORS[i] for i in range(5)]
labels = [f'Grade {i}\n{DR_GRADES[i]}' for i in range(5)]
axes[0].bar(labels, class_counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, (count, pct) in enumerate(zip(class_counts.values, class_pct.values)):
    axes[0].text(i, count + 30, f'{count}\n({pct}%)', ha='center', fontweight='bold', fontsize=10)
axes[0].set_ylabel('Count')
axes[0].set_title('APTOS 2019 — DR Grade Distribution', fontweight='bold')

# Pie chart
axes[1].pie(
    class_counts.values, labels=[DR_GRADES[i] for i in range(5)],
    colors=colors, autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Class Proportion', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Raw Image Exploration

### 4.1 Image Dimensions & Aspect Ratios

In [ ]:
# Sample a subset for speed
sample_df = df.sample(min(200, len(df)), random_state=RANDOM_SEED)

dims = []
for _, row in sample_df.iterrows():
    img_path = APTOS_TRAIN_IMAGES / f"{row['id_code']}.png"
    img = cv2.imread(str(img_path))
    if img is not None:
        h, w = img.shape[:2]
        dims.append({'id_code': row['id_code'], 'height': h, 'width': w,
                     'aspect_ratio': w / h, 'megapixels': h * w / 1e6})

dims_df = pd.DataFrame(dims)
print(f'Sampled {len(dims_df)} images')
dims_df[['height', 'width', 'aspect_ratio', 'megapixels']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(dims_df['width'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Width (px)')

axes[1].hist(dims_df['height'], bins=30, color='coral', edgecolor='white')
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Height (px)')

axes[2].hist(dims_df['aspect_ratio'], bins=30, color='mediumpurple', edgecolor='white')
axes[2].axvline(1.0, color='red', linestyle='--', label='Square')
axes[2].set_title('Aspect Ratio Distribution')
axes[2].set_xlabel('Width / Height')
axes[2].legend()

plt.tight_layout()
plt.show()

### 4.2 Sample Raw Images (One per Grade)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for grade in range(5):
    sample = df[df['diagnosis'] == grade].sample(1, random_state=RANDOM_SEED)
    img_path = APTOS_TRAIN_IMAGES / f"{sample.iloc[0]['id_code']}.png"
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    axes[grade].imshow(img_rgb)
    axes[grade].set_title(f'Grade {grade}: {DR_GRADES[grade]}',
                          fontweight='bold', color=DR_COLORS[grade])
    axes[grade].axis('off')

plt.suptitle('Raw Fundus Images — One per DR Grade', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Ben Graham Preprocessing Pipeline

The preprocessing has two stages:

1. **Circular Crop** — Detect the retina boundary via contour detection, crop to bounding box + 5% margin, resize to 512×512, apply circular mask (zero out black surround).

2. **Local Color Normalization** — Subtract the local average color using Gaussian blur:
   ```
   result = 4 × image − 4 × GaussianBlur(image, σ=width/30) + 128
   ```
   This removes illumination gradients while preserving lesion detail.

In [ ]:
# Demonstrate preprocessing on 3 images (different grades)
demo_grades = [0, 2, 4]
fig, axes = plt.subplots(len(demo_grades), 4, figsize=(20, 5 * len(demo_grades)))

for row_idx, grade in enumerate(demo_grades):
    sample = df[df['diagnosis'] == grade].sample(1, random_state=RANDOM_SEED)
    img_path = APTOS_TRAIN_IMAGES / f"{sample.iloc[0]['id_code']}.png"
    img = cv2.imread(str(img_path))

    # Stage 1: Raw
    raw_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[row_idx, 0].imshow(raw_rgb)
    axes[row_idx, 0].set_title(f'① Raw ({img.shape[1]}×{img.shape[0]})')

    # Stage 2: Circular crop only
    cropped = circular_crop(img, IMAGE_SIZE)
    crop_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
    axes[row_idx, 1].imshow(crop_rgb)
    axes[row_idx, 1].set_title(f'② Cropped ({IMAGE_SIZE}×{IMAGE_SIZE})')

    # Stage 3: Color normalization only (on cropped)
    normalized = local_color_normalization(cropped)
    norm_rgb = cv2.cvtColor(normalized, cv2.COLOR_BGR2RGB)
    axes[row_idx, 2].imshow(norm_rgb)
    axes[row_idx, 2].set_title('③ + Color Normalization')

    # Stage 4: Full pipeline
    full = ben_graham_preprocess(img, IMAGE_SIZE)
    full_rgb = cv2.cvtColor(full, cv2.COLOR_BGR2RGB)
    axes[row_idx, 3].imshow(full_rgb)
    axes[row_idx, 3].set_title('④ Full Pipeline')

    # Row label
    axes[row_idx, 0].set_ylabel(f'Grade {grade}\n{DR_GRADES[grade]}',
                                 fontsize=12, fontweight='bold', rotation=0,
                                 labelpad=80)

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Ben Graham Preprocessing — Step by Step',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Pixel Intensity Analysis

Compare pixel distributions **before** and **after** preprocessing.

In [ ]:
n_samples = 50
sample_ids = df.sample(n_samples, random_state=RANDOM_SEED)['id_code'].values

raw_means = {'R': [], 'G': [], 'B': []}
proc_means = {'R': [], 'G': [], 'B': []}

for id_code in sample_ids:
    img_path = APTOS_TRAIN_IMAGES / f'{id_code}.png'
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    # Raw (BGR → RGB)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    for ch, name in enumerate(['R', 'G', 'B']):
        raw_means[name].append(rgb[:, :, ch].mean())

    # Preprocessed
    proc = ben_graham_preprocess(img, IMAGE_SIZE)
    proc_rgb = cv2.cvtColor(proc, cv2.COLOR_BGR2RGB)
    for ch, name in enumerate(['R', 'G', 'B']):
        proc_means[name].append(proc_rgb[:, :, ch].mean())

print(f'Analyzed {len(raw_means["R"])} images')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

channel_colors = {'R': 'red', 'G': 'green', 'B': 'blue'}

for name in ['R', 'G', 'B']:
    axes[0].hist(raw_means[name], bins=20, alpha=0.5,
                 color=channel_colors[name], label=name, edgecolor='white')
axes[0].set_title('Raw — Mean Channel Intensity', fontweight='bold')
axes[0].set_xlabel('Mean Pixel Value')
axes[0].legend()

for name in ['R', 'G', 'B']:
    axes[1].hist(proc_means[name], bins=20, alpha=0.5,
                 color=channel_colors[name], label=name, edgecolor='white')
axes[1].set_title('After Ben Graham — Mean Channel Intensity', fontweight='bold')
axes[1].set_xlabel('Mean Pixel Value')
axes[1].legend()

plt.tight_layout()
plt.show()

print('\n📌 Notice how the preprocessed channels are centered around 128,')
print('   with much tighter spread — illumination variation is removed.')

## 7. Per-Grade Pixel Statistics

In [ ]:
grade_stats = []
for grade in range(5):
    grade_df = df[df['diagnosis'] == grade].sample(
        min(30, len(df[df['diagnosis'] == grade])), random_state=RANDOM_SEED
    )
    means, stds = [], []
    for _, row in grade_df.iterrows():
        img_path = APTOS_TRAIN_IMAGES / f"{row['id_code']}.png"
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        proc = ben_graham_preprocess(img, IMAGE_SIZE)
        proc_rgb = cv2.cvtColor(proc, cv2.COLOR_BGR2RGB)
        means.append(proc_rgb.mean())
        stds.append(proc_rgb.std())

    grade_stats.append({
        'grade': grade,
        'name': DR_GRADES[grade],
        'mean_intensity': np.mean(means),
        'std_intensity': np.mean(stds),
        'n_samples': len(means)
    })

stats_df = pd.DataFrame(grade_stats)
stats_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = range(5)
ax.bar(x, stats_df['mean_intensity'], yerr=stats_df['std_intensity'],
       color=[DR_COLORS[i] for i in range(5)], edgecolor='white',
       capsize=5, linewidth=1.5)
ax.set_xticks(x)
ax.set_xticklabels([f'Grade {i}\n{DR_GRADES[i]}' for i in range(5)])
ax.set_ylabel('Mean Pixel Intensity (after preprocessing)')
ax.set_title('Per-Grade Intensity Statistics', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Grid View — 3×5 Preprocessed Samples

In [ ]:
n_per_grade = 3
fig, axes = plt.subplots(n_per_grade, 5, figsize=(20, 4 * n_per_grade))

for grade in range(5):
    samples = df[df['diagnosis'] == grade].sample(
        n_per_grade, random_state=RANDOM_SEED
    )
    for row_idx, (_, row) in enumerate(samples.iterrows()):
        img_path = APTOS_TRAIN_IMAGES / f"{row['id_code']}.png"
        img = cv2.imread(str(img_path))
        proc = ben_graham_preprocess(img, IMAGE_SIZE)
        proc_rgb = cv2.cvtColor(proc, cv2.COLOR_BGR2RGB)

        axes[row_idx, grade].imshow(proc_rgb)
        axes[row_idx, grade].axis('off')
        if row_idx == 0:
            axes[row_idx, grade].set_title(
                f'Grade {grade}: {DR_GRADES[grade]}',
                fontweight='bold', color=DR_COLORS[grade], fontsize=12
            )

plt.suptitle('Preprocessed Fundus Images — 3 Samples per Grade',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 9. DataLoader Verification

Verify that the PyTorch `Dataset` and `DataLoader` work correctly with the preprocessing pipeline.

In [ ]:
from dataset import (
    DRDataset, get_train_val_split,
    get_train_transform, get_val_transform, create_dataloaders
)

# Split
train_df, val_df = get_train_val_split(df, val_fold=0)
print(f'Train: {len(train_df)},  Val: {len(val_df)}')
print(f'Train class dist:\n{train_df["diagnosis"].value_counts().sort_index()}')
print(f'\nVal class dist:\n{val_df["diagnosis"].value_counts().sort_index()}')

In [ ]:
# Create a small dataset and verify shapes
val_dataset = DRDataset(
    df=val_df, image_dir=str(APTOS_TRAIN_IMAGES),
    transform=get_val_transform(IMAGE_SIZE),
    preprocess=True, target_size=IMAGE_SIZE
)

img, label = val_dataset[0]
print(f'Image tensor shape: {img.shape}')     # (3, 512, 512)
print(f'Image dtype:        {img.dtype}')      # float32
print(f'Label:              {label} ({DR_GRADES[label]})')
print(f'Value range:        [{img.min():.3f}, {img.max():.3f}]')

In [ ]:
# Class weights for Focal Loss
train_dataset = DRDataset(
    df=train_df, image_dir=str(APTOS_TRAIN_IMAGES),
    transform=get_train_transform(IMAGE_SIZE),
    preprocess=True, target_size=IMAGE_SIZE
)

weights = train_dataset.get_class_weights()
print('Computed class weights (alpha for Focal Loss):')
for i in range(5):
    print(f'  Grade {i} ({DR_GRADES[i]:20s}): {weights[i]:.3f}')

## 10. Summary

| Aspect | Finding |
|--------|---------|
| **Dataset size** | ~3,662 images |
| **Class imbalance** | Grade 0 dominates (~50%), Grade 3/4 are rare (~5%) |
| **Image resolution** | Varies widely (need resize to 512×512) |
| **Preprocessing** | Ben Graham pipeline centers intensities around 128 with tight spread |
| **Data pipeline** | PyTorch Dataset + albumentations verified working |

**Next** → Notebook 2: Train a ResNet-50 baseline on this preprocessed data.